# Flood Event Inundation Mapping

This notebook uses the 2018 Labor Day flood event at the Wildcat Creek (near Manhattan, KS) as an example to demonstrate how to map a flood event with a tiled FLDPLN library.

## Import Modules and Packages

In [1]:
import time

# import the mapping and gauge modules from the fldpln package
from fldpln.mapping import *
from fldpln.gauge import *

## Setup Input Tiled Library and Output Folders

Here we setup the folder under which tiled libraries (organized as sub-folders) are located. We also setup the output folder under which a map sub-folder and a 'scratch' sub-folder will be created. The map sub-folder, which is specified later, contains all inundation depth maps. The scratch folder stores temporary files.

In [2]:
# tiled library folder
libFolder =  'E:/fldpln/workshop/devcon26/wildcat_10m_3dep' # wildcat

# libraries to be mapped, i.e., gauge stages will be snapped to the FSPs in those libraries.
libs2Map = ['tiledlib']

# Set mapping output folder
outputFolder = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps'

## Prepare Gauge Stage and Calculate Gauge Depth of Flow (DOF)

Here we obtain and process flood event stages recorded by stream gauges. The stage at a gauge typically (at least for USGS gauges) refers to difference from the gauge's datum, which is based on a certain vertical datum and is not necessary of the stream bed elevation. 

The stage in FLDPLN library, called Depth of Flow (DOF), refers to the height of water level from stream bed. In order to use the gauge stage in a FLDPLN library, we need to:
* Convert gauge stage to gauge stage elevation (i.e., water surface = gauge datum + stage) and make sure that gauge stage elevation and FLDPLN filled stream bed elevation are in the same vertical datum. 
* Calculate the depth of flow (DOF) at a FSP as the difference between the two elevation (DOF = gauge stage elevation - FSP bed elevation). 

The Wildcat Creek DEM and FLDPLN library are based on the NAVD88 vertical datum. So gauge stage elevations need to be based on the vertical datum too to calculate the DOFs at those gauges. 

### Gauge Stage from AHPS and USGS

In the United States, both USGS (through its [National Water Information System](https://waterdata.usgs.gov/nwis)) and NOAA/NWS (through its [National Water Prediction Service](https://water.noaa.gov), formerly called Advanced Hydrologic Prediction Services (AHPS)) maintain stream gauge records of past flood stages. 

There are three AHPS and USGS gauges ([WKCK1](https://water.noaa.gov/gauges/wkck1), [MWCK1](https://water.noaa.gov/gauges/MWCK1), [MSTK1](https://water.noaa.gov/gauges/MSTK1)) located on the Wildcat Creek that record the 2018 Labor Day flood event. Here we use the maximum flood event stage at those gauges to map the maximum inundation extent and depth of the event.

#### Flood Event Stage from AHPS Historic Crests

The flood stage for the 2018 Labor Day flood event in 2018 are available on [National Water Prediction Service](https://water.noaa.gov) as the historic crests at those gauges ([WKCK1](https://water.noaa.gov/gauges/wkck1), [MWCK1](https://water.noaa.gov/gauges/MWCK1), [MSTK1](https://water.noaa.gov/gauges/MSTK1)). 

An Excel file, wildcat_gauges_albers_meters.xlsx, has been prepared after all the USGS and NOAA/NWS gauges available in Kansas are retrieved and cleaned up. There are several sheets in the file which store both gauge information (for example, gauge datum) and the event stages with different gauge combinations. The key fields needed for flood inundation mapping from those gauges are: stationid, x, y, and stage_elevation.

Note that most USGS and AHPS gauge locations are in geographic coordinates (i.e., longitudes and latitudes) and their stages are measured in feet. So we need to make sure to convert gauge coordinates into the spatial reference used in the FLDPLN library and to convert gauge stages into the vertical unit of the library too.

In [8]:
# Wildcat Creek gauges
# gaugeStageFileName = 'wildcat_gauges.xlsx' # KS LiDAR DEM in UTM with vertical unit in feet
gaugeStageFile = 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/wildcat_gauges_albers_meters.xlsx' # 3DEP DEM in Albers with vertical unit in meters
# sheetName = 'ThreeGauges' # 3 gauges with the hightest stages during the flood event
sheetName = 'MWCK1' # the gauge used in NWS HECRAS-based inundation map library with a stage of 28 feet

# read gauge file
gaugeStages = pd.read_excel(gaugeStageFile, sheet_name=sheetName) 
# print(gaugeStages)

# Need to calculate gauge stage elevation if necessary!

# keep only necessary fields from gauges
keptFields = ['stationid','x','y','stage_elevation']
gaugeWithStageElevations = gaugeStages[keptFields]
print(gaugeWithStageElevations)

        stationid           x             y  stage_elevation
0  06879810,MWCK1 -54988.6141  1.796210e+06       325.819008


### Snap Gauges to FSPs and Calculate Gauge DOF

Here we snap the gauges to flood source pixels (FSPs), which are the stream pixels. We calculated the depth of flow/flood (DOF) at each snapped FSP as the difference between stage elevation and the stream bed elevation at the FSP. 

This process also identifies the FLDPLN libraries that the gauges are snapped to. This is necessary as the same gauge may be snapped to more than one library as the libraries may overlap and the FSPs in the overlay may have different locations! 

In [9]:
# snap gauges to FSPs on-the-fly
print('Snap gauges to FSPs ...')
print(f'Number of gauges: {len(gaugeWithStageElevations.index)}')

# snap the gauges to the FSPs in the selected libraries. Note that libraries may overlap in space and the same gauge may be snapped to multiple FSPs in different libraries.
# A snapped FSP is identified by 'lib_name' and FSP's ID together.
# Fields 'StrOrd','DsDist','SegId','FilledElev'are used for interpolating other FSP DOF
# Note that 'lib_name','FspX', 'FspY' together uniquely identify a FSP (as there are overlapping FSPs between libraries)!
gaugeFspDf = SnapGauges2Fsps(libFolder,libs2Map,gaugeWithStageElevations,snapDist=350,gaugeXField='x',gaugeYField='y',fspColumns=['FspX','FspY','StrOrd','DsDist','SegId','FilledElev']) 
print(gaugeFspDf)

# calculate gauge FSP's DOF
gaugeFspDf['Dof'] = gaugeFspDf['stage_elevation'] - gaugeFspDf['FilledElev']

# keep only necessary columns for gauge FSPs
gaugeFspDf = gaugeFspDf[['lib_name','FspX','FspY','StrOrd','DsDist','SegId','FilledElev','Dof']] # Note that 'lib_name','FspX', 'FspY' together uniquely identify a FSP!!!

# show info
print(f'Number of snapped gauge FSPs: {len(gaugeFspDf)}')
# Find libs where the gauges are snapped to, and they are the actual libs to map
libs2Map = gaugeFspDf['lib_name'].drop_duplicates().tolist()
print(f'Libraries gauges snapped to: {libs2Map}')
print(gaugeFspDf)

#
# save snapped gauges to CSV file for checking
# gaugeFspDf.to_csv(os.path.join(outputFolder, 'SnappedGauges.csv'), index=False)

Snap gauges to FSPs ...
Number of gauges: 1
   index       stationid           x             y  stage_elevation  \
0      0  06879810,MWCK1 -54988.6141  1.796210e+06       325.819008   

  d2NearestFsp          FspX            FspY     StrOrd        DsDist SegId  \
0     2.043149 -54988.205794  1796207.944565  1001003.0  16424.448763   9.0   

   FilledElev  lib_name  
0  318.953552  tiledlib  
Number of snapped gauge FSPs: 1
Libraries gauges snapped to: ['tiledlib']
   lib_name          FspX            FspY     StrOrd        DsDist SegId  \
0  tiledlib -54988.205794  1796207.944565  1001003.0  16424.448763   9.0   

   FilledElev       Dof  
0  318.953552  6.865456  


## Interpolate FSP DOF Between Gauges

Here we interpolate the DOF for the FSPs between any two gauge-snapped FSPs using their DOFs calculated in previous step. The interpolation uses stream orders and starts from low stream order (i.e., main streams) to high stream order (i.e., tributaries). Either horizontal or vertical (by default) interpolation can be used.

In [10]:
# Find libs with snapped gauges. They are the actual libs to map
libs2Map = gaugeFspDf['lib_name'].drop_duplicates().tolist()

# prepare the DF for storing interpolated FSP DOF
fspDof = pd.DataFrame(columns=['LibName','FspId','Dof'])

# prepare DFs for saving interpolated FSPs and their segment IDs
fspCols = fspInfoColumnNames + ['Dof']
segIdCols = ['SegId','LibName']
fsps = pd.DataFrame(columns=fspCols)
segIds =pd.DataFrame(columns=segIdCols)

# map each library
for libName in libs2Map:
    # interpolate DOF for the gauges
    # print('Interpolate FSP DOF using gauge DOF ...')
    # fspIdDof = InterpolateFspDofFromGauge(libFolder,libName,gaugeFspDf) # 'V' by default for vertical interpolation
    # fspIdDof = InterpolateFspDofFromGauge(libFolder,libName,gaugeFspDf,weightingType='H') # horizontal interpolation
    # fspIdDof = InterpolateFspDofFromGaugeThroughVolume(libFolder, libName, gaugeFspDf,netType='Filtered',dsPropSegNum=2 # has bugs?
    # fspIdDof = InterpolateFspDofFromGaugeThroughVolume(libFolder, libName, gaugeFspDf,netType='Full',dsPropSegNum=2) # volume-based interpolation
    fspIdDof = InterpolateCategoryFspDofFromGaugeThroughVolume(libFolder, libName, gaugeFspDf, upDist=3, dsDist=5) # Category FIM, 3 upstream segments and 5 downstream segments for interpolation
    fspIdDof['LibName'] = libName
    # fspDof = fspDof.append(fspIdDof[['LibName','FspId','Dof']], ignore_index=True)
    fspDof = pd.concat([fspDof,fspIdDof[['LibName','FspId','Dof']]], ignore_index=True)

    # Keep interpolated FSP DOF for saving later
    fspFile = os.path.join(libFolder, libName, fspInfoFileName)
    fspDf = pd.read_csv(fspFile) 
    fspDf = pd.merge(fspDf,fspDof,how='inner',on=['FspId'])
    # fsps = fsps.append(fspDf, ignore_index=True)
    fsps = pd.concat([fsps,fspDf], ignore_index=True)
    
    # Keep FSP segment IDs for saving later
    t =  pd.DataFrame(fspDf['SegId'].drop_duplicates().sort_values())
    t['LibName'] = libName
    # segIds = segIds.append(t, ignore_index=True)
    segIds = pd.concat([segIds,t], ignore_index=True)

# show interpolated FSPs with Dof
print(fspDof)

tiledlib has 1 gauged segments.
Processing gauged segment 9.0 ...
Segment 9.0 has DOF of 6.87 ft and volume of 3869997.75 acre-ft.
Segment 9.0 has 3 upstream segments within 3 segments: [(8, 3869997.74590706), (7, 3869997.74590706), (6, 3869997.74590706)]
Segment 9.0 has 5 downstream segments within 5 segments: [(10, 3869997.74590706), (11, 3869997.74590706), (12, 3869997.74590706), (13, 3869997.74590706), (14, 3869997.74590706)]
Segment 9.0 has 9 upstream and downstreamsegments: [(8, 3869997.74590706), (7, 3869997.74590706), (6, 3869997.74590706), (9.0, 3869997.74590706), (10, 3869997.74590706), (11, 3869997.74590706), (12, 3869997.74590706), (13, 3869997.74590706), (14, 3869997.74590706)]
Interpolated FSP DOF for segment 9.0 with 1619 FSPs.       FspId       Dof
0       904  6.816831
1       905  6.816831
2       906  6.816831
3       907  6.816831
4       908  6.816831
...     ...       ...
1614   2330  4.783099
1615   2331  4.783099
1616   2332  4.783099
1617   2333  4.783099
1618 

## Map Flood Inundation Depth


### Set Mapping Parameters

Setup the map folder which is under the output folder and contains all inundation depth maps. Additional settings include whether to mosaic tiles as single COG GeoTIFF file and whether using a Dask local cluster to speed up the mapping.

In [11]:
# set up map folder
outMapFolderName = 'mwck1_28'; # Wildcat Labor Day flood 

# Create folders for storing temp and output map files
outMapFolder, scratchFolder = CreateFolders(outputFolder,'scratch',outMapFolderName)

# whether mosaic tiles as a single COG
mosaicTiles = True #True #False

### Map Flood Inundation Depth

Here are create the flood depth map using the interpolated DOFs.

In [12]:
# show mapping info
print(f'Tiled FLDPLN library folder: {libFolder}')
print(f'Map folder: {outMapFolder}')
# Find libs needs mapping
libs2Map = fspDof['LibName'].drop_duplicates().tolist()
print(f'Libraries to map: {libs2Map}')

# check running time
startTimeAllLibs = time.time()

# dict to store lib processing time
libTime={}

# map each library
for libName in libs2Map:
    # check running time
    startTime = time.time()
    
    # select the FSPs within the lib
    fspIdDof = fspDof[fspDof['LibName']==libName][['FspId','Dof']]

    print(f'Map {libName} ...')
    tileTifs = MapFloodDepthWithTiles(libFolder,libName,'snappy',outMapFolder,fspIdDof,aoiExtent=None)
    print(f'Actual mapped tiles: {tileTifs}')

    # Mosaic all the tiles from a library into one tif file
    if mosaicTiles and not(tileTifs is None):
        print('Mosaic tile maps ...')
        mosaicTifName = libName+'_'+outMapFolderName+'.tif'
        # Simplest implementation, may crash with very large raster
        MosaicGtifs(outMapFolder,tileTifs,mosaicTifName,keepTifs=False)
    
    # check time
    endTime = time.time()
    usedTime = round((endTime-startTime)/60,3)
    libTime[libName] = usedTime
    # print(f'{libName} processing time (minutes):', usedTime)

# Show processing time
# Individual lib processing time
print('Individual library mapping time:', libTime)
# total time
endTimeAllLibs = time.time()
print('Total processing time (minutes):', round((endTimeAllLibs-startTimeAllLibs)/60,3))

Tiled FLDPLN library folder: E:/fldpln/workshop/devcon26/wildcat_10m_3dep
Map folder: E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\mwck1_28
Libraries to map: ['tiledlib']
Map tiledlib ...
Tiles need to be mapped: [6, 8, 9, 11, 12, 14]
Actual mapped tiles: ['E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mwck1_28\\tiledlib_tile_6.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mwck1_28\\tiledlib_tile_8.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mwck1_28\\tiledlib_tile_9.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mwck1_28\\tiledlib_tile_11.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mwck1_28\\tiledlib_tile_12.tif', 'E:/fldpln/workshop/devcon26/wildcat_10m_3dep/maps\\mwck1_28\\tiledlib_tile_14.tif']
Mosaic tile maps ...
Individual library mapping time: {'tiledlib': 0.012}
Total processing time (minutes): 0.012


## Compare Inundation Map with NWS FIM Library

Here we can generate an inundation map for the MWCK1 gauge with a specific stage and compare it with the corresponding NOAA/NWS FIM library map (nws_mwck1_1069.tif in the example data) which is retrieved from https://water.noaa.gov/gauges/mwck1.